# V7.2 (imgsz 960) — Vật thể nhỏ: baseline 960 → train → val gate → test (RTX 4060)

**Vì sao có thí nghiệm này:** kiểm thử MF04 (AI-ROB-04) cho thấy seed85 sót nhiều vật thể nhỏ: `broken_strand` cạnh ngắn 32–64 px (ở 640) recall 0.475, `insulator_broken` < 16 px recall 0.156. Thí nghiệm FO của V7.2 (notebook `fo_size_ablation`) đã hòa seed85, nên phía dữ liệu FO coi như xong.

**Thí nghiệm này chỉ đổi độ phân giải**, giữ nguyên dữ liệu và hyperparameter:
- Dữ liệu: **đúng dataset train của seed85** (`finding_nemo_human_reviewed_6cls`), val/test giữ nguyên.
- Fine-tune từ seed85 ở `imgsz=960`, batch 4 (Ultralytics cộng dồn gradient tới nbs=64 nên batch hiệu dụng không đổi). Các hyperparameter khác y hệt V7.1/V7.2.

**Bước 0 không cần train:** chạy chính seed85 ở 960. Nếu chỉ cần tăng độ phân giải lúc suy luận đã đủ thì khỏi train.

Gate so với **seed85 ở 640** (cấu hình đang deploy): PASS khi val mAP50-95 ≥ seed85 và FO AP50-95 không giảm. Test chỉ chạy cho bản PASS. Mỗi cell chạy lại an toàn: train dang dở thì resume, xong rồi thì bỏ qua.

In [ ]:
# Cấu hình chính
from pathlib import Path
import os, subprocess, sys, json, time

EXPERIMENT = 'finding_v7p2_211126'  # chung thư mục với thí nghiệm FO của V7.2
REMOTE_ROOT = f'stash://bunpmc/projects/{EXPERIMENT}'
SEED85_REMOTE = 'stash://bunpmc/projects/finding_seed/seed85/curated_yolo26m/weights/best.pt'

# Mỗi biến thể: imgsz và batch khi train. Muốn thử thêm 1024 thì bỏ comment dòng dưới.
VARIANTS = {
    'i960': dict(imgsz=960, batch=4),
    # 'i1024': dict(imgsz=1024, batch=4),
}
BASE_IMGSZ = 640       # seed85 và endpoint đang deploy
# Fine-tune từ một model đã hội tụ ở 640: 20 epoch, patience 8 (ở 960 mỗi epoch ~2,3× lâu hơn 640).
EPOCHS, PATIENCE, SEED, SYNC_EVERY = 20, 8, 85, 5
FO_TOLERANCE = 0.0
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def find_repo():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / 'benchmakring_transmission_line_defect_model').is_dir():
            return p
    raise RuntimeError('Mở notebook từ trong repo train')

ROOT = find_repo()
os.chdir(ROOT)
BENCH = ROOT / 'benchmakring_transmission_line_defect_model'
DATA_ROOT = BENCH / 'data' / 'finding_nemo_human_reviewed_6cls'
DATA_YAML = DATA_ROOT / 'data_local.yaml'
PROJECT = ROOT / 'runs' / EXPERIMENT
OUT = BENCH / 'results' / EXPERIMENT
SEED85 = ROOT / 'weights/v7p1_seed85_curated_yolo26m_best.pt'

PROJECT.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

def run_name(variant):
    return f'yolo26m_v7p2_{variant}'

print('ROOT :', ROOT)
print('DATA :', DATA_ROOT)
print('OUT  :', OUT)

## 1. Môi trường
Dùng cùng kernel `.venv` với notebook V7.1/V7.2. Nếu import lỗi, bỏ comment dòng `%pip`, chạy cell rồi restart kernel.

In [ ]:
# %pip install -U "ultralytics==8.4.172" python-dotenv pandas
import torch, ultralytics, yaml, pandas as pd
from ultralytics import YOLO
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass
assert torch.cuda.is_available(), 'Không thấy CUDA. Kiểm tra PyTorch CUDA/driver NVIDIA.'
print('Python     :', sys.executable)
print('Ultralytics:', ultralytics.__version__)
print('PyTorch    :', torch.__version__)
print('GPU        :', torch.cuda.get_device_name(0))
print('VRAM       :', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

## 1b. Hàm đánh giá (nhúng sẵn, không cần file script)
Code nhúng từ `benchmakring_transmission_line_defect_model/scripts/eval_v7p2_fo.py`: `model.val` cho số chính thức, confusion matrix, FO theo kích thước, recall theo kích thước mọi class, gallery ảnh FO bị sai, gate so với model đầu tiên. Gọi bằng `evaluate('--data', ..., '--model', 'ten=duong_dan.pt', ...)`; mỗi model có thể có `imgsz` riêng qua `--imgsz-for ten=960`.

In [ ]:
# fmt: off
"""Evaluate V7.2 candidates against seed85, with a foreign_object (FO) error breakdown.

For every model and split it writes:
  summary.csv / per_class.csv        official Ultralytics val metrics (same settings as V7.1 notebook)
  confusion_<model>_<split>.csv      GT class x predicted class (IoU >= --match-iou, one-to-one), plus missed / FP
  fo_outcomes_<model>_<split>.csv    one row per FO GT box: area bin, outcome, matched class, confidence, IoU
  fo_by_area.csv                     FO recall / FO->insulator_broken rate per area bin
  recall_by_size.csv                 recall per class and GT size (short side in a 640 reference)
  gallery/<model>_<split>/           images of FO GT boxes predicted as another class or missed
  gate.csv                           pass/fail against the baseline (only on val)

Test is evaluated only when --split test is given explicitly; choose checkpoints on val.

Example (Windows):
  python benchmakring_transmission_line_defect_model/scripts/eval_v7p2_fo.py ^
    --data benchmakring_transmission_line_defect_model/data/finding_nemo_6cls_v7p2_a_base_only/data_local.yaml ^
    --model seed85=weights/v7p1_seed85_curated_yolo26m_best.pt ^
    --model v7p1=runs/finding_v7p1_171126/yolo26m_finetune_v2_reviewed_181126/weights/best.pt ^
    --model v7p2b=runs/finding_v7p2/yolo26m_v7p2_b_fo_large/weights/best.pt ^
    --out benchmakring_transmission_line_defect_model/results/finding_v7p2/eval_val
"""
from __future__ import annotations

import argparse
import csv
import json
import time
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
FO, IB = 4, 1
AREA_BINS = ((0.0, 0.01, "small_lt1pct"), (0.01, 0.05, "medium_1to5pct"), (0.05, 1.01, "large_ge5pct"))


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--data", type=Path, required=True)
    p.add_argument("--model", action="append", required=True, help="name=path; the first one is the baseline")
    p.add_argument("--split", nargs="+", default=["val"], choices=["val", "test"])
    p.add_argument("--out", type=Path, required=True)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--imgsz-for", action="append", default=[],
                   help="name=size: evaluate one model at another image size (e.g. seed85_i960=960)")
    p.add_argument("--batch", type=int, default=8)
    p.add_argument("--device", default="0")
    p.add_argument("--conf", type=float, default=0.25, help="confidence for the confusion analysis")
    p.add_argument("--match-iou", type=float, default=0.5)
    p.add_argument("--gallery", type=int, default=60, help="max FO error images per model/split")
    p.add_argument("--fo-tolerance", type=float, default=0.0, help="allowed FO AP50-95 drop on val for the gate")
    return p.parse_args(argv)


def area_bin(area):
    return next(name for lo, hi, name in AREA_BINS if lo <= area < hi)


def read_gt(path: Path):
    out = []
    if path.is_file():
        for line in path.read_text(encoding="utf-8-sig").splitlines():
            p = line.split()
            if len(p) == 5:
                c, x, y, w, h = int(float(p[0])), *map(float, p[1:])
                out.append((c, x - w / 2, y - h / 2, x + w / 2, y + h / 2))
    return out


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match(gt, preds, thr):
    """Class-agnostic one-to-one matching by descending IoU. Returns {gt_idx: (pred_idx, iou)}."""
    pairs = sorted(((iou(g[1:], p[1:5]), gi, pi) for gi, g in enumerate(gt) for pi, p in enumerate(preds)),
                   reverse=True)
    used_g, used_p, result = set(), set(), {}
    for v, gi, pi in pairs:
        if v < thr:
            break
        if gi in used_g or pi in used_p:
            continue
        used_g.add(gi); used_p.add(pi); result[gi] = (pi, v)
    return result


def resolve_split(data_yaml: Path, split: str):
    cfg = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
    root = Path(cfg.get("path") or data_yaml.parent)
    if not root.is_absolute():
        root = (data_yaml.parent / root).resolve()
    images = root / cfg[split]  # e.g. <root>/images/val -> labels in <root>/labels/val
    labels = images.parent.parent / "labels" / images.name
    if not (images.is_dir() and labels.is_dir()):
        raise FileNotFoundError(f"{split}: expected {images} and {labels}")
    return images, labels


def draw(image_path: Path, gt_box, pred, label, dst: Path):
    with Image.open(image_path) as im:
        im = im.convert("RGB")
        W, H = im.size
        d = ImageDraw.Draw(im)
        lw = max(2, W // 400)
        if pred is not None:  # red = prediction, drawn first so the green GT stays visible on top
            d.rectangle([pred[1] * W, pred[2] * H, pred[3] * W, pred[4] * H], outline=(230, 30, 30), width=lw * 2)
        d.rectangle([gt_box[1] * W, gt_box[2] * H, gt_box[3] * W, gt_box[4] * H], outline=(0, 220, 0), width=lw)
        font = ImageFont.load_default(size=max(16, W // 50))
        d.rectangle(d.textbbox((8, 8), label, font=font), fill=(0, 0, 0))
        d.text((8, 8), label, fill=(255, 255, 0), font=font)
        im.thumbnail((1280, 1280))
        im.save(dst, quality=88)


SIZE_BINS = ((0, 16, "<16px"), (16, 32, "16-32px"), (32, 64, "32-64px"), (64, 1e9, ">=64px"))


def size_bin(short_side_640):
    return next(name for lo, hi, name in SIZE_BINS if lo <= short_side_640 < hi)


def analyze(model, name, split, images_dir, labels_dir, a, out, imgsz):
    images = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    conf_mat = defaultdict(Counter)  # gt class -> predicted class / "missed"
    fp = Counter()
    fo_rows = []
    size_hits = defaultdict(lambda: [0, 0])  # (class, size bin) -> [matched with right class, total]
    gallery = out / "gallery" / f"{name}_{split}"
    gallery.mkdir(parents=True, exist_ok=True)
    saved = 0
    for start in range(0, len(images), a.batch):
        chunk = images[start:start + a.batch]
        # rect=False: square letterbox, as in training and in the deployed endpoint.
        results = model.predict([str(p) for p in chunk], conf=a.conf, imgsz=imgsz, rect=False,
                                device=a.device, verbose=False)
        for img, r in zip(chunk, results):
            gt = read_gt(labels_dir / f"{img.stem}.txt")
            H, W = r.orig_shape
            b = r.boxes
            preds = [(int(c), *xy, float(s)) for c, xy, s in
                     zip(b.cls.tolist(), b.xyxyn.tolist(), b.conf.tolist())]
            m = match(gt, preds, a.match_iou)
            matched = {v[0] for v in m.values()}
            for pi, p in enumerate(preds):
                if pi not in matched:
                    fp[NAMES[p[0]]] += 1
            for gi, g in enumerate(gt):
                if gi in m:
                    pi, v = m[gi]
                    pc, pconf, pbox = NAMES[preds[pi][0]], preds[pi][5], preds[pi]
                else:
                    pc, pconf, v, pbox = "missed", "", "", None
                conf_mat[NAMES[g[0]]][pc] += 1
                # Size in a fixed 640 reference so every model bins the same GT box identically.
                side = min((g[3] - g[1]) * W, (g[4] - g[2]) * H) * 640 / max(W, H)
                hit = size_hits[(NAMES[g[0]], size_bin(side))]
                hit[0] += pc == NAMES[g[0]]
                hit[1] += 1
                if g[0] != FO:
                    continue
                area = (g[3] - g[1]) * (g[4] - g[2])
                fo_rows.append({"image": img.name, "area": round(area, 5), "area_bin": area_bin(area),
                                "outcome": pc, "confidence": pconf, "iou": round(v, 3) if v != "" else ""})
                if pc != "foreign_object" and saved < a.gallery:
                    draw(img, g, pbox, f"GT foreign_object -> {pc} {pconf if pconf == '' else round(pconf, 2)}",
                         gallery / f"{pc}__{img.stem}_{gi}.jpg")
                    saved += 1

    cols = NAMES + ["missed"]
    with (out / f"confusion_{name}_{split}.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["gt\\pred"] + cols)
        for g in NAMES:
            w.writerow([g] + [conf_mat[g][c] for c in cols])
        w.writerow(["background_fp"] + [fp[c] for c in NAMES] + [""])
    with (out / f"fo_outcomes_{name}_{split}.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["image", "area", "area_bin", "outcome", "confidence", "iou"])
        w.writeheader(); w.writerows(fo_rows)

    by_bin = []
    for *_, bn in AREA_BINS + (("", "", "all"),):
        rows = [r for r in fo_rows if bn == "all" or r["area_bin"] == bn]
        n = len(rows)
        oc = Counter(r["outcome"] for r in rows)
        by_bin.append({"model": name, "split": split, "area_bin": bn, "fo_gt": n,
                       "recall_at_conf": round(oc["foreign_object"] / n, 4) if n else "",
                       "as_insulator_broken": oc["insulator_broken"], "as_other_class":
                       n - oc["foreign_object"] - oc["insulator_broken"] - oc["missed"], "missed": oc["missed"]})
    ib_as_fo = conf_mat["insulator_broken"]["foreign_object"]
    print(f"  [{name}/{split}] FO->IB={conf_mat['foreign_object']['insulator_broken']} "
          f"IB->FO={ib_as_fo} FO missed={conf_mat['foreign_object']['missed']} FO FP={fp['foreign_object']}")
    size_rows = [{"model": name, "split": split, "class": c, "size_bin_640": b, "gt": n,
                  "recall_at_conf": round(t / n, 4)}
                 for c in NAMES for *_, b in SIZE_BINS for t, n in [size_hits.get((c, b), (0, 0))] if n]
    return by_bin, size_rows


def main(argv=None):
    a = parse_args(argv)
    models = []
    for spec in a.model:
        name, _, path = spec.partition("=")
        if not path or not Path(path).is_file():
            raise FileNotFoundError(f"--model must be name=existing_path, got {spec}")
        models.append((name, Path(path)))
    imgsz_for = {}
    for spec in a.imgsz_for:
        name, _, size = spec.partition("=")
        if name not in {n for n, _ in models} or not size.isdigit():
            raise ValueError(f"--imgsz-for must be model_name=size for a given --model, got {spec}")
        imgsz_for[name] = int(size)
    a.out.mkdir(parents=True, exist_ok=True)
    summary, per_class, fo_bins, size_rows = [], [], [], []

    for name, ckpt in models:
        model = YOLO(str(ckpt))
        imgsz = imgsz_for.get(name, a.imgsz)
        for split in a.split:
            print(f"\n===== {name} / {split} (imgsz {imgsz}) =====")
            m = model.val(data=str(a.data), split=split, imgsz=imgsz, batch=a.batch, device=a.device,
                          project=str(a.out / "runs"), name=f"{name}_{split}", exist_ok=True,
                          plots=True, save_json=False, verbose=False)
            summary.append({"model": name, "split": split, "imgsz": imgsz, "precision": float(m.box.mp), "recall": float(m.box.mr),
                            "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map)})
            for i, c in enumerate(m.box.ap_class_index.astype(int).tolist()):
                per_class.append({"model": name, "split": split, "class": m.names[c],
                                  "precision": float(m.box.p[i]), "recall": float(m.box.r[i]),
                                  "AP50": float(m.box.ap50[i]), "AP50_95": float(m.box.ap[i])})
            images_dir, labels_dir = resolve_split(a.data, split)
            bins, sizes = analyze(model, name, split, images_dir, labels_dir, a, a.out, imgsz)
            fo_bins += bins
            size_rows += sizes

    def write(rows, fname):
        if not rows:
            return
        with (a.out / fname).open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0]))
            w.writeheader(); w.writerows(rows)
    write(summary, "summary.csv"); write(per_class, "per_class.csv"); write(fo_bins, "fo_by_area.csv")
    write(size_rows, "recall_by_size.csv")

    gate = []
    base = models[0][0]
    if "val" in a.split:
        S = {(r["model"], r["split"]): r for r in summary}
        P = {(r["model"], r["split"], r["class"]): r["AP50_95"] for r in per_class}
        for name, _ in models[1:]:
            d_map = S[name, "val"]["mAP50_95"] - S[base, "val"]["mAP50_95"]
            d_m50 = S[name, "val"]["mAP50"] - S[base, "val"]["mAP50"]
            # A class without GT in the split has no AP for any model: nothing to compare.
            present = [c for c in NAMES if (name, "val", c) in P and (base, "val", c) in P]
            d_fo = (P[name, "val", "foreign_object"] - P[base, "val", "foreign_object"]
                    if "foreign_object" in present else 0.0)
            better = [c for c in present if P[name, "val", c] > P[base, "val", c]]
            ok = d_map >= 0 and d_fo >= -a.fo_tolerance
            gate.append({"model": name, "baseline": base, "d_mAP50": round(d_m50, 4), "d_mAP50_95": round(d_map, 4),
                         "d_FO_AP50_95": round(d_fo, 4), "classes_improved": "|".join(better),
                         "gate": "PASS" if ok else "FAIL"})
        if gate:
            write(gate, "gate.csv")
            print("\nVal gate vs", base)
            for g in gate:
                print(f"  {g['model']:>12}: {g['gate']}  dmAP50-95={g['d_mAP50_95']:+.4f} "
                      f"dFO={g['d_FO_AP50_95']:+.4f} improved={g['classes_improved']}")
    (a.out / "eval_results.json").write_text(json.dumps({
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"), "data": str(a.data), "models": [list(map(str, m)) for m in models],
        "imgsz_for": imgsz_for, "summary": summary, "per_class": per_class, "fo_by_area": fo_bins,
        "recall_by_size": size_rows, "gate": gate}, indent=2), encoding="utf-8")
    print("\nSaved:", a.out)


def evaluate(*argv):
    """Notebook entry point: evaluate("--data", path, "--model", "a=b.pt", ...)."""
    main([str(x) for x in argv])

print('evaluate() sẵn sàng')

## 2. Camber và checkpoint seed85
Cần `CAMBER_API_KEY` trong `.env`. Notebook V7.2 đã tải seed85 thì cell này chỉ kiểm tra.

In [ ]:
def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'):
        return
    env_file = ROOT / '.env'
    if env_file.is_file():
        for line in env_file.read_text(encoding='utf-8').splitlines():
            key, sep, value = line.partition('=')
            if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
                os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39)); return
    raise RuntimeError('Thiếu CAMBER_API_KEY trong environment hoặc .env của repo')

def wsl_path(path: Path):
    path = path.resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'

CAMBER = '/home/dotru/.camber/bin/camber'
load_camber_key()
CAMBER_ENV = os.environ.copy()
wslenv = [x for x in CAMBER_ENV.get('WSLENV', '').split(':') if x]
if 'CAMBER_API_KEY' not in wslenv: wslenv.append('CAMBER_API_KEY')
CAMBER_ENV['WSLENV'] = ':'.join(wslenv)

def camber(*parts, check=True):
    converted = [wsl_path(x) if isinstance(x, Path) else str(x) for x in parts]
    p = subprocess.run(['wsl', '--', CAMBER, 'stash', *converted], env=CAMBER_ENV,
                       capture_output=True, text=True, encoding='utf-8', errors='replace')
    if check and p.returncode:
        raise RuntimeError((p.stdout or p.stderr)[-3000:])
    return p

def ensure_remote_dir(uri):
    if camber('ls', uri, check=False).returncode != 0:
        made = camber('mkdir', uri, check=False)
        if made.returncode != 0 and camber('ls', uri, check=False).returncode != 0:
            raise RuntimeError((made.stdout or made.stderr)[-1200:])

def sync_file(local: Path, remote: str):
    if local.is_file():
        camber('cp', local, remote)

auth = subprocess.run(['wsl', '--', CAMBER, 'me'], env=CAMBER_ENV, capture_output=True,
                      text=True, encoding='utf-8', errors='replace')
assert auth.returncode == 0, (auth.stdout or auth.stderr)[-1200:]
print('Camber OK')
ensure_remote_dir('stash://bunpmc/projects'); ensure_remote_dir(REMOTE_ROOT)

if not SEED85.is_file():
    SEED85.parent.mkdir(parents=True, exist_ok=True)
    print('Downloading', SEED85_REMOTE); camber('cp', SEED85_REMOTE, SEED85)
assert SEED85.stat().st_size > 40_000_000, SEED85
print(SEED85.relative_to(ROOT), round(SEED85.stat().st_size / 2**20, 1), 'MiB')

## 3. Dataset của seed85
Kiểm tra đủ ảnh/label từng split và ghi `data_local.yaml` với đường dẫn tuyệt đối của máy này.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
EXPECTED = {'train': 17903, 'val': 2271, 'test': 1931}
for split, n in EXPECTED.items():
    images = [p for p in (DATA_ROOT / 'images' / split).iterdir() if p.suffix.lower() in IMAGE_EXTS]
    labels = list((DATA_ROOT / 'labels' / split).glob('*.txt'))
    missing = {p.stem for p in images} - {p.stem for p in labels}
    assert not missing, f'{split}: {len(missing)} ảnh thiếu label, ví dụ {sorted(missing)[:3]}'
    assert len(images) == n, f'{split}: {len(images)} ảnh, mong đợi {n} (không đúng dataset seed85?)'
    print(f'{split}: {len(images)} images, {len(labels)} labels')
DATA_YAML.write_text(
    'path: ' + DATA_ROOT.resolve().as_posix() + '\n'
    'train: images/train\nval: images/val\ntest: images/test\n'
    'names:\n  0: insulator\n  1: insulator_broken\n  2: insulator_flashover\n'
    '  3: bird_nest\n  4: foreign_object\n  5: broken_strand\n', encoding='utf-8')
for stale in (DATA_ROOT / 'labels').glob('*.cache'):
    stale.unlink()  # cache tạo từ máy/đường dẫn khác
print('Dataset:', DATA_YAML)

## 4. Bước 0: seed85 ở 960, không train
So seed85 ở 640 (đang deploy) với chính seed85 ở 960 trên val. Bảng recall theo kích thước cho biết vật nhỏ có được bắt thêm không. Nếu `seed85_i960` đã PASS gate thì chỉ cần đổi `IMAGE_SIZE` của endpoint, không cần train.

In [ ]:
BASE0 = OUT / 'i960_step0_val'
if not (BASE0 / 'eval_results.json').is_file():
    evaluate('--data', DATA_YAML, '--model', f'seed85={SEED85}',
           '--model', f'seed85_i960={SEED85}', '--imgsz-for', 'seed85_i960=960',
           '--split', 'val', '--fo-tolerance', FO_TOLERANCE, '--out', BASE0)

def size_view(folder, classes=('broken_strand', 'insulator_broken', 'insulator_flashover', 'foreign_object')):
    s = pd.read_csv(folder / 'recall_by_size.csv')
    s = s[s['class'].isin(classes)]
    order = {'<16px': 0, '16-32px': 1, '32-64px': 2, '>=64px': 3}
    t = s.pivot_table(index=['class', 'size_bin_640'], columns='model', values='recall_at_conf')
    n = s.drop_duplicates(['class', 'size_bin_640']).set_index(['class', 'size_bin_640'])['gt']
    t.insert(0, 'gt', n)
    return t.sort_index(key=lambda i: i.map(order) if i.name == 'size_bin_640' else i).round(3)

display(pd.read_csv(BASE0 / 'gate.csv'))
display(pd.read_csv(BASE0 / 'summary.csv'))
display(pd.read_csv(BASE0 / 'per_class.csv').pivot(index='class', columns='model', values='AP50_95').round(4))
display(size_view(BASE0))

## 5. Fine-tune seed85 ở 960
Mỗi biến thể là một run riêng trong `runs/finding_v7p2_211126/` (tên `yolo26m_v7p2_i960`). Có `last.pt` thì resume; có `DONE.json` thì bỏ qua; cứ 5 epoch đồng bộ Camber.

Nếu gặp `CUDA out of memory`: restart kernel, giảm `batch` của biến thể xuống 2 ở cell cấu hình, chạy lại từ đầu (run chưa có `last.pt` sẽ bắt đầu lại).

In [ ]:
import gc

def sync_run(run_dir, remote, epoch=None):
    ensure_remote_dir(remote); ensure_remote_dir(remote + '/weights')
    for rel in ('results.csv', 'args.yaml', 'DONE.json', 'weights/last.pt', 'weights/best.pt'):
        sync_file(run_dir / rel, f'{remote}/{rel}')
    if epoch:
        sync_file(run_dir / f'weights/epoch{epoch}.pt', f'{remote}/weights/epoch{epoch}.pt')
    print(f'[CAMBER SYNC] epoch={epoch} -> {remote}', flush=True)

def try_restore(remote, local):
    if local.exists():
        return
    local.parent.mkdir(parents=True, exist_ok=True)
    if camber('cp', remote, local, check=False).returncode == 0 and local.exists():
        print('[RESTORED]', local)

def train_variant(variant, cfg):
    name = run_name(variant)
    run_dir, remote = PROJECT / name, f'{REMOTE_ROOT}/{name}'
    if (run_dir / 'DONE.json').is_file():
        print(f'[SKIP] {variant} đã train xong'); return
    last = run_dir / 'weights/last.pt'
    for rel in ('weights/last.pt', 'results.csv', 'args.yaml'):
        try_restore(f'{remote}/{rel}', run_dir / rel)
    gc.collect(); torch.cuda.empty_cache()
    model = YOLO(str(last if last.is_file() else SEED85))
    clock = time.time()

    def on_fit_epoch_end(trainer):
        m = trainer.metrics or {}
        f = lambda k: f"{float(m.get(k, float('nan'))):.4f}"
        print(f'\n[V7.2 {variant}] epoch {trainer.epoch + 1}/{trainer.epochs} | '
              f'val mAP50={f("metrics/mAP50(B)")} | mAP50-95={f("metrics/mAP50-95(B)")} | '
              f'{(time.time() - clock) / 60:.1f} min', flush=True)

    def on_model_save(trainer):
        e = trainer.epoch + 1
        if e % SYNC_EVERY == 0 or e == trainer.epochs:
            sync_run(run_dir, remote, epoch=e)

    model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
    model.add_callback('on_model_save', on_model_save)
    if last.is_file():
        print('RESUME:', last)
        model.train(resume=True, data=str(DATA_YAML.resolve()))
    else:
        print(f'START {variant} từ seed85: imgsz={cfg["imgsz"]} batch={cfg["batch"]}')
        model.train(
            data=str(DATA_YAML), epochs=EPOCHS, imgsz=cfg['imgsz'], batch=cfg['batch'], device=0,
            optimizer='AdamW', lr0=1e-4, lrf=0.01, cos_lr=True, warmup_epochs=1.0,
            momentum=0.937, weight_decay=5e-4,
            mosaic=1.0, close_mosaic=10, fliplr=0.5, flipud=0.0, translate=0.1, scale=0.5,
            degrees=0.0, shear=0.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.4,
            amp=True, seed=SEED, deterministic=True, workers=2, cache=False,
            project=str(PROJECT), name=name, exist_ok=True,
            save=True, save_period=SYNC_EVERY, plots=True, patience=PATIENCE, val=True, verbose=True,
        )
    assert (run_dir / 'weights/best.pt').is_file(), run_dir
    (run_dir / 'DONE.json').write_text(json.dumps({
        'variant': variant, **cfg, 'data': str(DATA_YAML), 'init': str(SEED85),
        'finished_at': time.strftime('%Y-%m-%d %H:%M:%S')}, indent=2), encoding='utf-8')
    sync_run(run_dir, remote)
    del model; gc.collect(); torch.cuda.empty_cache()

for v, cfg in VARIANTS.items():
    train_variant(v, cfg)
print('Đã train xong:', [v for v in VARIANTS if (PROJECT / run_name(v) / 'DONE.json').is_file()])

## 6. Val gate so với seed85 ở 640
Mỗi model được đánh giá ở đúng `imgsz` của nó. `seed85_i960` (không train) cũng được đưa vào gate. Test chưa được đụng ở bước này.

In [ ]:
trained = [v for v in VARIANTS if (PROJECT / run_name(v) / 'DONE.json').is_file()]
CANDIDATES = {'seed85_i960': (SEED85, 960)}
CANDIDATES.update({v: (PROJECT / run_name(v) / 'weights/best.pt', VARIANTS[v]['imgsz']) for v in trained})
VAL = OUT / 'i960_eval_val'  # eval_val/ là kết quả thí nghiệm FO, không ghi đè
args = ['--model', f'seed85={SEED85}']
for name, (ckpt, size) in CANDIDATES.items():
    args += ['--model', f'{name}={ckpt}', '--imgsz-for', f'{name}={size}']
evaluate('--data', DATA_YAML, *args, '--split', 'val',
       '--fo-tolerance', FO_TOLERANCE, '--out', VAL)

def size_view(folder, classes=('broken_strand', 'insulator_broken', 'insulator_flashover', 'foreign_object')):
    s = pd.read_csv(folder / 'recall_by_size.csv')
    s = s[s['class'].isin(classes)]
    order = {'<16px': 0, '16-32px': 1, '32-64px': 2, '>=64px': 3}
    t = s.pivot_table(index=['class', 'size_bin_640'], columns='model', values='recall_at_conf')
    n = s.drop_duplicates(['class', 'size_bin_640']).set_index(['class', 'size_bin_640'])['gt']
    t.insert(0, 'gt', n)
    return t.sort_index(key=lambda i: i.map(order) if i.name == 'size_bin_640' else i).round(3)

gate = pd.read_csv(VAL / 'gate.csv')
display(gate)
display(pd.read_csv(VAL / 'summary.csv'))
display(pd.read_csv(VAL / 'per_class.csv').pivot(index='class', columns='model', values='AP50_95').round(4))
display(size_view(VAL))
PASSED = [m for m in gate[gate.gate == 'PASS'].model if m in CANDIDATES]
print('PASS val gate:', PASSED or 'không có — giữ seed85 ở 640')

## 7. Test cố định (chỉ cho bản PASS)
Chỉ chạy một lần. Không dùng kết quả test để chỉnh tham số rồi chạy lại.

In [ ]:
if not PASSED:
    print('Không có bản nào PASS val gate. Giữ seed85 ở 640, không chạy test.')
else:
    TEST = OUT / 'i960_eval_test'
    args = ['--model', f'seed85={SEED85}']
    for name in PASSED:
        ckpt, size = CANDIDATES[name]
        args += ['--model', f'{name}={ckpt}', '--imgsz-for', f'{name}={size}']
    evaluate('--data', DATA_YAML, *args, '--split', 'test', '--out', TEST)
    s = pd.read_csv(TEST / 'summary.csv').set_index('model')
    display(s)
    cols = ['precision', 'recall', 'mAP50', 'mAP50_95']
    display((s.drop(index='seed85')[cols] - s.loc['seed85', cols]).round(4))
    display(pd.read_csv(TEST / 'per_class.csv').pivot(index='class', columns='model', values='AP50_95').round(4))
    display(size_view(TEST))

## 8. Upload kết quả lên Camber

In [ ]:
for sub in ('i960_step0_val', 'i960_eval_val', 'i960_eval_test'):
    folder = OUT / sub
    if not folder.is_dir():
        continue
    remote = f'{REMOTE_ROOT}/evaluation/{sub}'
    ensure_remote_dir(f'{REMOTE_ROOT}/evaluation'); ensure_remote_dir(remote)
    for f in sorted(folder.glob('*.csv')) + sorted(folder.glob('*.json')):
        sync_file(f, f'{remote}/{f.name}')
    print('Uploaded', sub, '->', remote)